# Lab 1: NetworkX

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bonsai-lab-polimi/PyNet2Plan-lab/blob/main/labs/lab1_exercises.ipynb)

The NetworkX part of Lab 1 (handout 2, *Networks as NetworkX graphs*) in the form of a notebook.
Part 1 is a short introduction to NetworkX with the network of the lab; Part 2 contains the
exercises, and the sections it mentions are those of handout 2. The network is Spain-7, with 40
wavelengths on every link, and the traffic is the matrix of the lab (24 demands, 229
wavelengths). Exercises 1 to 5 are done in the lab; 6 and 7 are optional.

**Self-contained.** The notebook needs only `networkx` and `matplotlib`, which Google Colab already
has: open it with the link above, or upload the file in Colab (*File > Upload notebook*), or open
it in Jupyter or VS Code. The data of the lab (the networks and traffic matrices of the folder
`data`) and the few functions of PyNet2Plan used by the exercises are in the first two cells. The
exercises are the same functions as in `labs/lab1_exercises.py`: work in the form you prefer.

**How to work.** Run the three setup cells (data, toolbox, tools), then the cells of Part 1 in
order. In Part 2, run the cell of the helpers; each exercise is then a function: write its code
where the comments indicate, delete the line `raise NotCompletedError(...)`, run the cell of the
function and then the cell below it, which runs the exercise.

**Saving your work.** Colab does not save the changes in the original notebook: use
*File > Save a copy in Drive*, or *File > Download > Download .ipynb*.

## Setup

The data of the lab, the toolbox and two tools of the notebook: `run(exercise)` runs an exercise of Part 2, `draw(H, paths)` draws a network and some paths on a map.

In [ ]:
# @title Data of the lab: networks and traffic matrices of the folder data (run this cell)
# Topology files: name, capacity of every link (wavelengths per fiber), nodes (name, lat, lon),
# links (origin, destination, length in km[, fibers if more than 1]). Traffic files: demands
# (id, origin, destination, wavelengths).
TOPOLOGIES = {
    "data/topologies/spain7.json": {
        "name": "Spain-7", "capacity": 40,
        "nodes": [
            ("Madrid", 40.4168, -3.7038),
            ("Barcelona", 41.3874, 2.1686),
            ("Valencia", 39.4699, -0.3763),
            ("Sevilla", 37.3891, -5.9845),
            ("Zaragoza", 41.6488, -0.8891),
            ("Malaga", 36.7213, -4.4214),
            ("Murcia", 37.9922, -1.1307),
        ],
        "links": [
            ("Madrid", "Zaragoza", 272.94), ("Madrid", "Sevilla", 390.21), ("Madrid", "Valencia", 302.56),
            ("Barcelona", "Zaragoza", 256.21), ("Barcelona", "Valencia", 303.06), ("Valencia", "Barcelona", 303.06),
            ("Valencia", "Murcia", 176.86), ("Valencia", "Madrid", 302.56), ("Sevilla", "Malaga", 157.33),
            ("Sevilla", "Madrid", 390.21), ("Zaragoza", "Madrid", 272.94), ("Zaragoza", "Barcelona", 256.21),
            ("Malaga", "Murcia", 323.34), ("Malaga", "Sevilla", 157.33), ("Murcia", "Valencia", 176.86),
            ("Murcia", "Malaga", 323.34),
        ],
    },
    "data/topologies/nsfnet14.json": {
        "name": "NSFNET-14", "capacity": 40,
        "nodes": [
            ("Seattle", 47.6062, -122.3321),
            ("Palo Alto", 37.4419, -122.143),
            ("San Diego", 32.7157, -117.1611),
            ("Salt Lake City", 40.7608, -111.891),
            ("Boulder", 40.015, -105.2705),
            ("Houston", 29.7604, -95.3698),
            ("Lincoln", 40.8136, -96.7026),
            ("Champaign", 40.1164, -88.2434),
            ("Pittsburgh", 40.4406, -79.9959),
            ("Atlanta", 33.749, -84.388),
            ("Ann Arbor", 42.2808, -83.743),
            ("Ithaca", 42.444, -76.5019),
            ("Princeton", 40.3573, -74.6672),
            ("College Park", 38.9807, -76.9369),
        ],
        "links": [
            ("Seattle", "Palo Alto", 1130.33), ("Seattle", "San Diego", 1712.05), ("Seattle", "Champaign", 2831.22),
            ("Palo Alto", "Seattle", 1130.33), ("Palo Alto", "San Diego", 693.81), ("Palo Alto", "Salt Lake City", 957.77),
            ("San Diego", "Seattle", 1712.05), ("San Diego", "Palo Alto", 693.81), ("San Diego", "Houston", 2093.86),
            ("Salt Lake City", "Palo Alto", 957.77), ("Salt Lake City", "Boulder", 566.68), ("Salt Lake City", "Ann Arbor", 2338.82),
            ("Boulder", "Salt Lake City", 566.68), ("Boulder", "Houston", 1452.47), ("Boulder", "Lincoln", 730.48),
            ("Houston", "San Diego", 2093.86), ("Houston", "Boulder", 1452.47), ("Houston", "Atlanta", 1128.27),
            ("Houston", "College Park", 1972.43), ("Lincoln", "Boulder", 730.48), ("Lincoln", "Champaign", 719.53),
            ("Champaign", "Seattle", 2831.22), ("Champaign", "Lincoln", 719.53), ("Champaign", "Pittsburgh", 700.32),
            ("Pittsburgh", "Champaign", 700.32), ("Pittsburgh", "Atlanta", 839.58), ("Pittsburgh", "Ithaca", 366.61),
            ("Pittsburgh", "Princeton", 451.26), ("Atlanta", "Houston", 1128.27), ("Atlanta", "Pittsburgh", 839.58),
            ("Ann Arbor", "Salt Lake City", 2338.82), ("Ann Arbor", "Ithaca", 595.04), ("Ann Arbor", "Princeton", 787.1),
            ("Ithaca", "Pittsburgh", 366.61), ("Ithaca", "Ann Arbor", 595.04), ("Ithaca", "College Park", 386.84),
            ("Princeton", "Pittsburgh", 451.26), ("Princeton", "Ann Arbor", 787.1), ("Princeton", "College Park", 247.31),
            ("College Park", "Houston", 1972.43), ("College Park", "Ithaca", 386.84), ("College Park", "Princeton", 247.31),
        ],
    },
    "data/topologies/cost239.json": {
        "name": "COST239", "capacity": 40,
        "nodes": [
            ("Amsterdam", 52.3676, 4.9041),
            ("Berlin", 52.52, 13.405),
            ("Brussels", 50.8503, 4.3517),
            ("Copenhagen", 55.6761, 12.5683),
            ("London", 51.5074, -0.1278),
            ("Luxembourg", 49.6116, 6.1319),
            ("Milan", 45.4642, 9.19),
            ("Paris", 48.8566, 2.3522),
            ("Prague", 50.0755, 14.4378),
            ("Vienna", 48.2082, 16.3738),
            ("Zurich", 47.3769, 8.5417),
        ],
        "links": [
            ("Amsterdam", "London", 357.89), ("Amsterdam", "Berlin", 576.09), ("Amsterdam", "Brussels", 172.97),
            ("Amsterdam", "Luxembourg", 318.26), ("Amsterdam", "Copenhagen", 620.72), ("Berlin", "Amsterdam", 576.09),
            ("Berlin", "Copenhagen", 355.15), ("Berlin", "Prague", 281.13), ("Berlin", "Vienna", 523.55),
            ("Berlin", "Paris", 877.46), ("Brussels", "London", 320.63), ("Brussels", "Amsterdam", 172.97),
            ("Brussels", "Paris", 263.98), ("Brussels", "Luxembourg", 187.09), ("Brussels", "Milan", 697.85),
            ("Copenhagen", "London", 955.56), ("Copenhagen", "Amsterdam", 620.72), ("Copenhagen", "Berlin", 355.15),
            ("Copenhagen", "Prague", 635.21), ("London", "Amsterdam", 357.89), ("London", "Brussels", 320.63),
            ("London", "Paris", 343.56), ("London", "Copenhagen", 955.56), ("Luxembourg", "Amsterdam", 318.26),
            ("Luxembourg", "Brussels", 187.09), ("Luxembourg", "Paris", 286.95), ("Luxembourg", "Prague", 597.51),
            ("Luxembourg", "Zurich", 305.38), ("Milan", "Brussels", 697.85), ("Milan", "Paris", 639.57),
            ("Milan", "Zurich", 218.41), ("Milan", "Vienna", 625.5), ("Paris", "London", 343.56),
            ("Paris", "Berlin", 877.46), ("Paris", "Brussels", 263.98), ("Paris", "Luxembourg", 286.95),
            ("Paris", "Zurich", 487.88), ("Paris", "Milan", 639.57), ("Prague", "Berlin", 281.13),
            ("Prague", "Copenhagen", 635.21), ("Prague", "Luxembourg", 597.51), ("Prague", "Vienna", 250.87),
            ("Prague", "Zurich", 526.14), ("Vienna", "Berlin", 523.55), ("Vienna", "Milan", 625.5),
            ("Vienna", "Prague", 250.87), ("Vienna", "Zurich", 592.07), ("Zurich", "Luxembourg", 305.38),
            ("Zurich", "Paris", 487.88), ("Zurich", "Milan", 218.41), ("Zurich", "Prague", 526.14),
            ("Zurich", "Vienna", 592.07),
        ],
    },
    "data/topologies/real/top22_geant2.json": {
        "name": "GEANT2 (real topology #22)", "capacity": 40,
        "nodes": [
            ("Lisbon", 38.7077507, -9.1365919),
            ("Madrid", 40.4167047, -3.7035825),
            ("Paris", 48.8588897, 2.320041021720077),
            ("London", 51.5074456, -0.1277653),
            ("Dublin", 53.3493795, -6.2605593),
            ("Brussels", 50.8465573, 4.351697),
            ("Amsterdam", 52.3730796, 4.8924534),
            ("Luxembourg", 49.6112768, 6.129799),
            ("Bern", 46.9484742, 7.4521749),
            ("Rome", 41.8933203, 12.4829321),
            ("Valetta", 35.8989818, 14.5136759),
            ("Athens", 37.98, 23.72),
            ("Nicosia", 35.1746503, 33.3638783),
            ("Ankara", 39.9207759, 32.8540497),
            ("Sofia", 42.96, 23.25),
            ("Bucharest", 44.4361414, 26.1027202),
            ("Budapest", 47.4978789, 19.0402383),
            ("Zagreb", 45.8130967, 15.9772795),
            ("Vienna", 48.2083537, 16.3725042),
            ("Bratislava", 48.1516988, 17.1093063),
            ("Ljubljana", 46.0500268, 14.5069289),
            ("Prague", 50.0874654, 14.4212535),
            ("Berlin", 52.5170365, 13.3888599),
            ("Copenhagen", 55.6867243, 12.5700724),
            ("Oslo", 59.9133301, 10.7389701),
            ("Stockholm", 59.3251172, 18.0710935),
            ("Helsinki", 60.1674881, 24.9427473),
            ("Tallinn", 59.4372155, 24.7453688),
            ("Vilnius", 54.6870458, 25.2829111),
            ("Moscow", 55.7505412, 37.6174782),
            ("Riga", 56.9493977, 24.1051846),
            ("Warsaw", 52.2337172, 21.07143223563649),
            ("Manchester", 53.4794892, -2.2451148),
            ("Cardiff", 51.4816546, -3.1791934),
            ("Mariehamn", 60.102423, 19.94126),
        ],
        "links": [
            ("Lisbon", "Madrid", 754.35), ("Lisbon", "London", 1982.99), ("Madrid", "Lisbon", 754.35),
            ("Madrid", "Paris", 1500.0), ("Madrid", "Rome", 1703.76), ("Madrid", "Bern", 1500.0),
            ("Paris", "London", 513.25), ("Paris", "Madrid", 1500.0), ("Paris", "Bern", 656.17),
            ("Paris", "Luxembourg", 433.43, 2), ("London", "Lisbon", 1982.99), ("London", "Manchester", 392.93),
            ("London", "Cardiff", 316.88), ("London", "Brussels", 481.11), ("London", "Paris", 513.25),
            ("Dublin", "Manchester", 399.72), ("Dublin", "Cardiff", 441.87), ("Brussels", "London", 481.11),
            ("Brussels", "Amsterdam", 260.7), ("Amsterdam", "Brussels", 260.7), ("Amsterdam", "Copenhagen", 932.52),
            ("Amsterdam", "Berlin", 863.61), ("Amsterdam", "Valetta", 2477.68), ("Luxembourg", "Paris", 433.43, 2),
            ("Luxembourg", "Berlin", 901.5), ("Bern", "Paris", 656.17), ("Bern", "Madrid", 1500.0),
            ("Bern", "Berlin", 1127.12), ("Bern", "Rome", 1033.95), ("Rome", "Madrid", 1703.76),
            ("Rome", "Bern", 1033.95), ("Rome", "Vienna", 1148.31), ("Rome", "Athens", 1576.5),
            ("Rome", "Nicosia", 2446.82), ("Rome", "Valetta", 1033.89), ("Valetta", "Amsterdam", 2477.68),
            ("Valetta", "Rome", 1033.89), ("Athens", "Rome", 1576.5), ("Athens", "Nicosia", 1372.5),
            ("Athens", "Sofia", 828.0), ("Athens", "Vienna", 1603.0), ("Nicosia", "Rome", 2446.82),
            ("Nicosia", "Athens", 1372.5), ("Ankara", "Bucharest", 1123.37), ("Ankara", "Sofia", 1302.0),
            ("Sofia", "Athens", 828.0), ("Sofia", "Ankara", 1302.0), ("Sofia", "Bucharest", 423.0),
            ("Sofia", "Budapest", 903.75), ("Bucharest", "Ankara", 1123.37), ("Bucharest", "Sofia", 423.0),
            ("Bucharest", "Budapest", 964.39), ("Budapest", "Sofia", 903.75), ("Budapest", "Bucharest", 964.39),
            ("Budapest", "Bratislava", 242.16), ("Budapest", "Zagreb", 449.31), ("Zagreb", "Budapest", 449.31),
            ("Zagreb", "Ljubljana", 175.08), ("Vienna", "Rome", 1148.31), ("Vienna", "Athens", 1603.0),
            ("Vienna", "Ljubljana", 417.6), ("Vienna", "Berlin", 785.47), ("Vienna", "Bratislava", 82.48),
            ("Bratislava", "Budapest", 242.16), ("Bratislava", "Vienna", 82.48), ("Bratislava", "Prague", 436.23),
            ("Ljubljana", "Zagreb", 175.08), ("Ljubljana", "Vienna", 417.6), ("Prague", "Bratislava", 436.23),
            ("Prague", "Warsaw", 781.92), ("Prague", "Berlin", 419.28), ("Berlin", "Amsterdam", 863.61),
            ("Berlin", "Luxembourg", 901.5), ("Berlin", "Bern", 1127.12), ("Berlin", "Vienna", 785.47),
            ("Berlin", "Prague", 419.28), ("Berlin", "Warsaw", 783.33), ("Berlin", "Copenhagen", 534.71),
            ("Berlin", "Moscow", 2012.44), ("Copenhagen", "Amsterdam", 932.52), ("Copenhagen", "Berlin", 534.71),
            ("Copenhagen", "Tallinn", 1253.43), ("Copenhagen", "Moscow", 1950.25), ("Copenhagen", "Stockholm", 781.29),
            ("Copenhagen", "Oslo", 723.44), ("Oslo", "Copenhagen", 723.44), ("Oslo", "Stockholm", 625.89),
            ("Stockholm", "Copenhagen", 781.29), ("Stockholm", "Oslo", 625.89), ("Stockholm", "Mariehamn", 203.84),
            ("Stockholm", "Helsinki", 594.0), ("Helsinki", "Mariehamn", 415.44), ("Helsinki", "Stockholm", 594.0),
            ("Tallinn", "Copenhagen", 1253.43), ("Tallinn", "Riga", 418.74), ("Vilnius", "Riga", 393.13),
            ("Vilnius", "Warsaw", 584.91), ("Moscow", "Berlin", 2012.44), ("Moscow", "Copenhagen", 1950.25),
            ("Riga", "Tallinn", 418.74), ("Riga", "Vilnius", 393.13), ("Warsaw", "Prague", 781.92),
            ("Warsaw", "Berlin", 783.33), ("Warsaw", "Vilnius", 584.91), ("Manchester", "London", 392.93),
            ("Manchester", "Dublin", 399.72), ("Cardiff", "London", 316.88), ("Cardiff", "Dublin", 441.87),
            ("Mariehamn", "Stockholm", 203.84), ("Mariehamn", "Helsinki", 415.44),
        ],
    },
}

TRAFFIC_FILES = {
    "data/traffic/spain7_traffic_split_vs_unsplit.csv": [
        (0, "Sevilla", "Barcelona", 8), (1, "Barcelona", "Madrid", 15), (2, "Madrid", "Sevilla", 10),
        (3, "Murcia", "Valencia", 5), (4, "Zaragoza", "Barcelona", 12), (5, "Madrid", "Valencia", 8),
        (6, "Malaga", "Barcelona", 9), (7, "Malaga", "Valencia", 6), (8, "Valencia", "Malaga", 5),
        (9, "Madrid", "Malaga", 16), (10, "Zaragoza", "Malaga", 14), (11, "Sevilla", "Malaga", 13),
        (12, "Valencia", "Zaragoza", 7), (13, "Barcelona", "Valencia", 13), (14, "Barcelona", "Zaragoza", 5),
        (15, "Barcelona", "Murcia", 6), (16, "Sevilla", "Murcia", 14), (17, "Sevilla", "Zaragoza", 15),
        (18, "Murcia", "Sevilla", 4), (19, "Zaragoza", "Sevilla", 10), (20, "Sevilla", "Valencia", 9),
        (21, "Murcia", "Zaragoza", 5), (22, "Malaga", "Sevilla", 5), (23, "Valencia", "Madrid", 15),
    ],
    "data/traffic/spain7_traffic.csv": [
        (0, "Madrid", "Barcelona", 6), (1, "Madrid", "Valencia", 6), (2, "Madrid", "Sevilla", 6),
        (3, "Madrid", "Zaragoza", 1), (4, "Madrid", "Malaga", 4), (5, "Madrid", "Murcia", 3),
        (6, "Barcelona", "Madrid", 4), (7, "Barcelona", "Valencia", 1), (8, "Barcelona", "Sevilla", 2),
        (9, "Barcelona", "Zaragoza", 3), (10, "Barcelona", "Malaga", 2), (11, "Barcelona", "Murcia", 3),
        (12, "Valencia", "Madrid", 1), (13, "Valencia", "Barcelona", 1), (14, "Valencia", "Sevilla", 5),
        (15, "Valencia", "Zaragoza", 3), (16, "Valencia", "Malaga", 4), (17, "Valencia", "Murcia", 5),
        (18, "Sevilla", "Madrid", 3), (19, "Sevilla", "Barcelona", 1), (20, "Sevilla", "Valencia", 5),
        (21, "Sevilla", "Zaragoza", 4), (22, "Sevilla", "Malaga", 4), (23, "Sevilla", "Murcia", 4),
        (24, "Zaragoza", "Madrid", 6), (25, "Zaragoza", "Barcelona", 3), (26, "Zaragoza", "Valencia", 5),
        (27, "Zaragoza", "Sevilla", 3), (28, "Zaragoza", "Malaga", 4), (29, "Zaragoza", "Murcia", 4),
        (30, "Malaga", "Madrid", 2), (31, "Malaga", "Barcelona", 1), (32, "Malaga", "Valencia", 3),
        (33, "Malaga", "Sevilla", 5), (34, "Malaga", "Zaragoza", 4), (35, "Malaga", "Murcia", 3),
        (36, "Murcia", "Madrid", 4), (37, "Murcia", "Barcelona", 2), (38, "Murcia", "Valencia", 2),
        (39, "Murcia", "Sevilla", 2), (40, "Murcia", "Zaragoza", 5), (41, "Murcia", "Malaga", 2),
    ],
}

CITIES = {
    "Bilbao": (43.263, -2.935),
    "Lisbon": (38.7223, -9.1393),
    "Porto": (41.1579, -8.6291),
    "Paris": (48.8566, 2.3522),
}

In [ ]:
# @title Toolbox: the functions of PyNet2Plan used by the exercises (run this cell)
# They behave as network.* and traffic.* of PyNet2Plan, read the data embedded above
# instead of the files, and write files that the graphical interface of PyNet2Plan opens.
import csv
import itertools
import json
import math
import os
import random
import types
from collections import namedtuple

import matplotlib.pyplot as plt
import networkx as nx

ROOT = "."                                       # paths such as ROOT/data/... refer to the data above
TOPOLOGY = os.path.join(ROOT, "data", "topologies", "spain7.json")
TRAFFIC = os.path.join(ROOT, "data", "traffic", "spain7_traffic_split_vs_unsplit.csv")
OUT = os.path.join("results", "lab1_exercises")  # files written by the exercises (Files panel in Colab)


class NotCompletedError(Exception):
    """Raised by the parts of the exercises that are still to be written."""


class Demand(namedtuple("Demand", "id origin destination traffic")):
    """A demand: ``traffic`` wavelengths from ``origin`` to ``destination``."""

    def __str__(self):
        return f"d{self.id}: {self.origin}->{self.destination} ({self.traffic} λ)"


def _embedded(path, files):
    key = os.path.normpath(path).replace(os.sep, "/")
    if key not in files:
        raise FileNotFoundError(f"{path}: not among the files of this notebook ({', '.join(files)})")
    return files[key]


def load_topology(path):
    """The network of a topology file (embedded above) as a networkx.DiGraph: nodes with lat
    and lon, links with capacity (wavelengths per fiber), fibers and length (km)."""
    data = _embedded(path, TOPOLOGIES)
    G = nx.DiGraph(name=data["name"])
    for name, lat, lon in data["nodes"]:
        G.add_node(name, lat=lat, lon=lon, up=True)
    for u, v, length, *fibers in data["links"]:
        G.add_edge(u, v, capacity=data["capacity"], fibers=fibers[0] if fibers else 1, length=length, up=True)
    return G


def load_traffic(path, G=None):
    """The demands of a traffic file (embedded above); with G, their nodes are checked."""
    demands = [Demand(*row) for row in _embedded(path, TRAFFIC_FILES)]
    for d in demands:
        if G is not None and (d.origin not in G or d.destination not in G):
            raise ValueError(f"demand {d}: unknown node")
    return demands


def total_traffic(demands):
    return sum(d.traffic for d in demands)


def describe(G):
    """One line describing the network, as in PyNet2Plan."""
    links = [a for _, _, a in G.edges(data=True)]
    caps = [a.get("capacity", 0) for a in links] or [0]
    fib = [a.get("fibers", 1) for a in links] or [0]
    up = sum(1 for a in links if a.get("up", True))
    cap = f"{min(caps)}" if min(caps) == max(caps) else f"{min(caps)}-{max(caps)}"
    fibers = "" if max(fib) <= 1 else (f", {min(fib)}" + (f"-{max(fib)}" if min(fib) != max(fib) else "")
                                       + " fibers per link")
    return (f"{G.graph.get('name', '')}: {G.number_of_nodes()} nodes, {G.number_of_edges()} links ({up} up), "
            f"capacity {cap} λ{' per fiber' if fibers else ''}{fibers}")


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres."""
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin(math.radians(lat2 - lat1) / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return 2 * 6371.0088 * math.asin(min(1.0, math.sqrt(a)))


def add_city(G, city, name=None, allow_online=False):
    """Adds a node at the coordinates of ``city`` (one of CITIES); returns its name."""
    if city not in CITIES:
        raise KeyError(f"{city}: unknown city; known: {', '.join(CITIES)} (or use G.add_node(name, lat=..., lon=...))")
    lat, lon = CITIES[city]
    name = name or city
    G.add_node(name, lat=lat, lon=lon, up=True)
    return name


def link_length(G, u, v):
    """Great-circle km between two nodes with lat and lon, otherwise the distance of x and y."""
    a, b = G.nodes[u], G.nodes[v]
    if a.get("lat") is not None and b.get("lat") is not None:
        return round(haversine_km(a["lat"], a["lon"], b["lat"], b["lon"]), 2) or 1.0
    return round(math.hypot(a["x"] - b["x"], a["y"] - b["y"]), 3) or 1.0


def add_link(G, origin, destination, capacity=40, length=None, bidirectional=False, fibers=1):
    """Adds the link origin -> destination (and the opposite one with bidirectional=True)."""
    for u, v in [(origin, destination)] + ([(destination, origin)] if bidirectional else []):
        if G.has_edge(u, v):
            raise ValueError(f"the link {u}->{v} already exists")
        G.add_edge(u, v, capacity=capacity, fibers=fibers,
                   length=link_length(G, u, v) if length is None else float(length), up=True)


def random_traffic(G, num_demands, min_traffic=1, max_traffic=8, seed=0):
    """num_demands demands between distinct random node pairs, each of a random number of
    wavelengths in [min_traffic, max_traffic]; the same seed gives the same demands."""
    rnd = random.Random(seed)
    pairs = [(a, b) for a in G.nodes for b in G.nodes if a != b]
    demands = []
    for i in range(min(num_demands, len(pairs))):          # partial Fisher-Yates shuffle
        j = i + int(rnd.random() * (len(pairs) - i))
        pairs[i], pairs[j] = pairs[j], pairs[i]
        traffic = min_traffic + int(rnd.random() * (max_traffic - min_traffic + 1))
        demands.append(Demand(i, pairs[i][0], pairs[i][1], traffic))
    return demands


def save_topology(G, path):
    """Saves the network as a PyNet2Plan topology file (File > Open topology... in the tool)."""
    nodes = []
    for n, a in G.nodes(data=True):
        item = {"name": n}
        item.update({k: a[k] for k in ("lat", "lon", "x", "y") if a.get(k) is not None})
        item["up"] = bool(a.get("up", True))
        nodes.append(item)
    links = []
    for u, v, a in G.edges(data=True):
        item = {"origin": u, "destination": v, "capacity": int(a.get("capacity", 40))}
        if int(a.get("fibers", 1)) != 1:
            item["fibers"] = int(a["fibers"])
        item.update(length=a.get("length"), up=bool(a.get("up", True)))
        links.append(item)
    data = {"format": "pynet2plan-topology", "version": 2, "name": G.graph.get("name", "Untitled network"),
            "nodes": nodes, "links": links}
    with open(path, "w", encoding="utf-8") as fh:
        json.dump(data, fh, indent=2, ensure_ascii=False)


def save_traffic(demands, path):
    """Saves the demands as a PyNet2Plan traffic file (Traffic > Open traffic matrix... in the tool)."""
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["id", "origin", "destination", "traffic"])
        for d in demands:
            w.writerow([d.id, d.origin, d.destination, d.traffic])


def from_networkx(H, capacity=40, name=None, size=1000.0):
    """A network made by a NetworkX generator (e.g. nx.waxman_graph) as a PyNet2Plan network:
    two links per edge, positions scaled to a size x size square, lengths from the positions."""
    pos = nx.get_node_attributes(H, "pos")
    if len(pos) != H.number_of_nodes():
        pos = nx.spring_layout(H, seed=0)
    xs, ys = [float(pos[n][0]) for n in H], [float(pos[n][1]) for n in H]
    span = max(max(xs) - min(xs), max(ys) - min(ys)) or 1.0
    G = nx.DiGraph(name=name or f"{type(H).__name__} ({H.number_of_nodes()} nodes)")
    for n, x, y in zip(H, xs, ys):
        G.add_node(str(n), x=round((x - min(xs)) / span * size, 1), y=round((y - min(ys)) / span * size, 1),
                   up=True)
    for u, v in H.edges():
        if u != v and not G.has_edge(str(u), str(v)):
            add_link(G, str(u), str(v), capacity=capacity, bidirectional=not H.is_directed())
    return G


network = types.SimpleNamespace(load_topology=load_topology, describe=describe, add_city=add_city,
                                add_link=add_link, save_topology=save_topology, from_networkx=from_networkx)
traffic = types.SimpleNamespace(load_traffic=load_traffic, total_traffic=total_traffic,
                                random_traffic=random_traffic, save_traffic=save_traffic)

G = network.load_topology(TOPOLOGY)
D = traffic.load_traffic(TRAFFIC, G)
print(network.describe(G))
print(f"{len(D)} demands, {traffic.total_traffic(D)} wavelengths")

In [ ]:
# Tools of the notebook (given): run an exercise, draw a network and some paths on a map
def run(exercise):
    """Runs an exercise on G and D; says which TODO is still missing."""
    print(exercise.__doc__.split("\n")[0])
    try:
        exercise(G, D)
    except NotCompletedError as e:
        print(f"  not completed yet: {e}")


def draw(H, paths=(), title=None, separate=False):
    """Draws H with longitude and latitude as coordinates, and each path of ``paths`` in its own
    colour; with separate=True, each path on its own map."""
    pos = {n: (H.nodes[n]["lon"], H.nodes[n]["lat"]) for n in H.nodes}
    above = {n: (x, y + 0.12) for n, (x, y) in pos.items()}         # labels above the nodes
    U = H.to_undirected()
    groups = [[p] for p in paths] if separate and paths else [list(paths)]
    fig, axes = plt.subplots(1, len(groups), figsize=(5.2 * len(groups) if separate else 6.5, 5), squeeze=False)
    for i, (ax, group) in enumerate(zip(axes[0], groups)):
        nx.draw_networkx_edges(U, pos, ax=ax, edge_color="#9e9e9e", width=1.5)
        for k, path in enumerate(group):
            color = ("#eb6834", "#1baf7a", "#2a78d6", "#7b4fd1")[(i + k) % 4]
            nx.draw_networkx_edges(U, pos, edgelist=list(zip(path, path[1:])), ax=ax, edge_color=color,
                                   width=4, label=" > ".join(path))
        nx.draw_networkx_nodes(H, pos, ax=ax, node_size=90, node_color="#2a78d6")
        nx.draw_networkx_labels(H, above, ax=ax, font_size=11, verticalalignment="bottom")
        ax.margins(0.12)
        ax.set_axis_off()
        if separate:
            ax.set_title(" > ".join(group[0]), fontsize=10)
        elif group:                                                  # legend below the map
            ax.legend(loc="upper center", bbox_to_anchor=(0.5, 0.0), fontsize=10, frameon=False)
    if separate:
        fig.suptitle(title or H.graph.get("name", ""), fontsize=13)
    else:
        axes[0][0].set_title(title or H.graph.get("name", ""), fontsize=13)
    plt.show()
    plt.close(fig)

# Part 1. A short introduction to NetworkX

NetworkX is the Python library used by PyNet2Plan to represent networks. This part presents the
calls needed in the exercises, with the network of the lab; it follows sections 1 to 7 of
handout 2. Run the cells in order and read their output.

## 1. Graphs, nodes and links

NetworkX represents a graph as a Python object: `nx.Graph()` is an undirected graph and
`nx.DiGraph()` a directed one. An optical network is a directed graph, because a pair of fibres
between two cities gives two links, one per direction, each with its own capacity and load.
Nodes and links can have attributes, which are stored in dictionaries.

In [ ]:
T = nx.DiGraph(name="Tiny")                    # an empty directed graph
T.add_node("A", lat=45.46, lon=9.19)           # a node with two attributes
T.add_node("B", lat=45.07, lon=7.69)
T.add_node("C", lat=44.41, lon=8.93)
T.add_edge("A", "B", capacity=40, fibers=1, length=126.0)   # the link A -> B ...
T.add_edge("B", "A", capacity=40, fibers=1, length=126.0)   # ... and the opposite one
T.add_edge("A", "C", capacity=10, fibers=2, length=118.0)   # 2 fibres of 10 wavelengths

print("nodes:", list(T.nodes))
print("node A:", T.nodes["A"])
print("links:", list(T.edges))
print("link A->C:", T.edges["A", "C"])
print("is there a link C->A?", T.has_edge("C", "A"))
print(T.number_of_nodes(), "nodes and", T.number_of_edges(), "links")

The capacity of a link, in wavelengths, is the number of fibres times the wavelengths per
fibre: for A→C, `T.edges["A", "C"]["fibers"] * T.edges["A", "C"]["capacity"]` = 20. In PyNet2Plan
every network is a `DiGraph` of this kind; the function `add_link(G, u, v, capacity=40,
bidirectional=True)` of the toolbox adds the two directions at once and computes the length from
the coordinates.

## 2. The network of the lab

`G`, loaded in the toolbox cell, is Spain-7: 7 cities and 8 pairs of fibres, that is 16 links,
with 40 wavelengths on every link. The links leaving and entering a node are given by
`G.out_edges(n)` and `G.in_edges(n)`, and their number by `G.out_degree(n)` and `G.in_degree(n)`.

In [ ]:
print(G.graph["name"], "-", G.number_of_nodes(), "nodes,", G.number_of_edges(), "links")
print("Madrid:", G.nodes["Madrid"])
print("links leaving Madrid:", list(G.out_edges("Madrid")))
print("links entering Madrid:", list(G.in_edges("Madrid")))
print("out-degree:", dict(G.out_degree()))

A loop over `G.edges(data=True)` gives every link with its attributes:

In [ ]:
print(f"{'link':22s} {'fibers':>6s} {'capacity':>8s} {'length (km)':>12s}")
for u, v, a in sorted(G.edges(data=True)):
    print(f"{u + ' -> ' + v:22s} {a['fibers']:6d} {a['capacity']:8d} {a['length']:12.1f}")
draw(G)

## 3. Shortest paths

`nx.shortest_path(G, s, t)` returns the path with the fewest links (hops) from s to t, as a list
of nodes. With `weight="length"` it returns the path of minimum total length, in kilometres.
`nx.path_weight(G, path, weight="length")` adds up an attribute along a path.

In [ ]:
s, t = "Sevilla", "Valencia"
p_hops = nx.shortest_path(G, s, t)
p_km = nx.shortest_path(G, s, t, weight="length")
for label, p in (("fewest hops", p_hops), ("shortest km", p_km)):
    print(f"{label}: {' > '.join(p)}  ({len(p) - 1} hops, {nx.path_weight(G, p, weight='length'):.0f} km)")
draw(G, [p_hops, p_km], title=f"{s} to {t}: fewest hops and shortest length")

The two criteria give different paths: the path through Madrid has one hop less, the path along
the coast is 35 km shorter. Without s and t, `nx.shortest_path_length(G)` gives the number of
hops between every pair of nodes, as a dictionary of dictionaries:

In [ ]:
hops = dict(nx.shortest_path_length(G))
print("hops from Zaragoza:", hops["Zaragoza"])
print("hops from Sevilla to Barcelona:", hops["Sevilla"]["Barcelona"])

## 4. The k shortest paths

The route formulations of Lab 2 choose the path of each demand among a few candidate paths.
`nx.shortest_simple_paths(G, s, t, weight="length")` generates all the paths without loops from
s to t, from the shortest to the longest, one at a time; `islice` takes the first k of them
without computing the others.

In [ ]:
from itertools import islice

paths = list(islice(nx.shortest_simple_paths(G, "Zaragoza", "Malaga", weight="length"), 3))
for i, p in enumerate(paths, 1):
    print(f"path {i}: {' > '.join(p)}  ({len(p) - 1} hops, {nx.path_weight(G, p, weight='length'):.0f} km)")
draw(G, paths, title="The 3 shortest paths from Zaragoza to Malaga", separate=True)

## 5. Connectivity and failures

A directed graph is strongly connected when every node can reach every other node. A fibre cut
removes the two links of a pair; it is simulated on a copy of the network, so that `G` does not
change. On the undirected version of the graph, `nx.bridges` lists the edges whose failure
disconnects the network and `nx.edge_connectivity` gives the smallest number of edges whose
failure does.

In [ ]:
U = G.to_undirected()
print("strongly connected:", nx.is_strongly_connected(G))
print("bridges:", list(nx.bridges(U)) or "none", "- edge connectivity:", nx.edge_connectivity(U))

F = G.copy()                                    # the network after the cut of Madrid-Valencia
F.remove_edges_from([("Madrid", "Valencia"), ("Valencia", "Madrid")])
print("Madrid -> Murcia before the cut:", " > ".join(nx.shortest_path(G, "Madrid", "Murcia")))
print("Madrid -> Murcia after the cut :", " > ".join(nx.shortest_path(F, "Madrid", "Murcia")))
print("still strongly connected:", nx.is_strongly_connected(F))

## 6. Demands

`D`, also loaded in the toolbox cell, is the traffic matrix of the lab: a list of demands, each
with an id, an origin, a destination and a traffic in wavelengths. The matrix was generated with
`traffic.random_traffic` and a seed: the same seed always gives the same demands, so a random
matrix can be reproduced by anybody.

In [ ]:
print(len(D), "demands,", traffic.total_traffic(D), "wavelengths")
for d in D[:4]:
    print(" ", d, "- origin", d.origin, "- traffic", d.traffic)
print("random_traffic(G, 24, 4, 16, seed=9) gives the same demands:",
      traffic.random_traffic(G, 24, 4, 16, seed=9) == D)

## 7. Two bounds from the graph

If the capacities were unlimited, every demand would follow a path with the fewest hops, and
the wavelengths used, summed over all the links, would be the traffic of each demand times its
number of hops. No routing can use fewer wavelengths, so this value is a lower bound for the
optimum of the formulations of handout 3.

In [ ]:
lower = sum(d.traffic * hops[d.origin][d.destination] for d in D)
print("fewest-hop routing:", lower, "wavelengths")

The capacities also have a lower bound. The traffic from a set S of nodes to the other nodes
must cross the links that leave S, so with C wavelengths on every link,
C ≥ (traffic leaving S) / (number of links leaving S) for every set S.
`nx.edge_boundary(G, S)` lists the links leaving S, and `itertools.combinations` gives every set
of k nodes.

In [ ]:
S = {"Madrid", "Barcelona", "Valencia", "Zaragoza", "Sevilla"}     # all but Malaga and Murcia
links = list(nx.edge_boundary(G, S))
need = sum(d.traffic for d in D if d.origin in S and d.destination not in S)
print(f"{need} wavelengths leave S on the links {links}: C >= {math.ceil(need / len(links))}")

best = None
for size in range(1, G.number_of_nodes()):
    for S in itertools.combinations(G.nodes, size):
        bound = math.ceil(sum(d.traffic for d in D if d.origin in S and d.destination not in S)
                          / len(list(nx.edge_boundary(G, S))))
        if best is None or bound > best[0]:
            best = (bound, sorted(S))
print("most restrictive set:", best[1], "-> C >=", best[0])
draw(G, [["Sevilla", "Malaga"], ["Valencia", "Murcia"]], title="The two links entering Malaga and Murcia")

## 8. Generating networks

NetworkX generates many kinds of graphs: rings, grids, random graphs, and Waxman graphs, whose
nodes are placed at random in a square and connected with a probability that decreases with
their distance. These graphs are undirected; `network.from_networkx` turns one into a network of
PyNet2Plan, with two links per edge and lengths taken from the positions, which can be saved with
`network.save_topology` and opened in the tool.

In [ ]:
generated = {"ring": nx.cycle_graph(8), "grid": nx.grid_2d_graph(3, 4),
             "Waxman": nx.waxman_graph(12, beta=0.7, alpha=0.35, seed=7),
             "random": nx.gnm_random_graph(10, 16, seed=3)}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (kind, H) in zip(axes, generated.items()):
    pos = nx.get_node_attributes(H, "pos") or nx.spring_layout(H, seed=0)
    nx.draw(H, pos, ax=ax, node_size=60, node_color="#2a78d6", edge_color="#9e9e9e")
    ax.set_title(f"{kind}: {H.number_of_nodes()} nodes, {H.number_of_edges()} edges\n"
                 f"connected: {nx.is_connected(H)}", fontsize=11)
plt.show()
plt.close(fig)
W = network.from_networkx(generated["Waxman"], capacity=40, name="Waxman 12")
print(network.describe(W))

These are all the calls needed in Part 2. The complete tour, with more comments, is the script
`labs/lab1_networkx_tour.py` described in handout 2.

# Part 2. Exercises

## Helpers (given)

The same helper functions as in `labs/lab1_exercises.py`.

In [ ]:
def fiber_cut(G, u, v):
    """A copy of G without the two directions of the link u-v (a fibre cut)."""
    H = G.copy()
    H.remove_edges_from([(u, v), (v, u)])
    return H


def fibers(G):
    """Every pair of opposite links once: ('Madrid', 'Zaragoza'), ..."""
    seen = []
    for u, v in G.edges:
        if (u, v) not in seen and (v, u) not in seen:
            seen.append((u, v))
    return seen


def fewest_hop_bound(G, D):
    """Wavelengths used if every demand follows a path with the fewest hops (handout 2, section 7)."""
    hops = dict(nx.shortest_path_length(G))
    return sum(d.traffic * hops[d.origin][d.destination] for d in D)


def km(G, path):
    return nx.path_weight(G, path, weight="length")

## The NetworkX calls you need

All of them are explained in handout 2 and used in `labs/lab1_networkx_tour.py`.

| call | result |
|---|---|
| `nx.shortest_path(G, s, t)` | path with the fewest hops, as a list of nodes |
| `nx.shortest_path(G, s, t, weight="length")` | shortest path in kilometres |
| `nx.shortest_path_length(G, s, t)` | number of hops of the fewest-hop path |
| `nx.path_weight(G, path, weight="length")` | length of a path in kilometres (helper `km(G, path)`) |
| `nx.is_strongly_connected(H)` | `True` if every node can reach every other node |
| `G.nodes`, `G.edges`, `G.edges[u, v]["length"]` | nodes, links and attributes of a link |
| `G.out_edges(n)`, `G.in_edges(n)` | links leaving and entering node n |
| `itertools.combinations(nodes, k)` | every set of k nodes |
| `d.origin`, `d.destination`, `d.traffic` | a demand of `D` |

The files mentioned in the exercises (`data/...`) are those embedded in the first cell, and
`network.*` and `traffic.*` are the functions of the toolbox. The tool `draw(H, [path1, path2])`
shows a network and some paths on a map (`separate=True`: one map per path).

## Exercise 1

Paths from Sevilla to Valencia with the fewest hops and in kilometres, before and after the cut of the fibre Madrid-Valencia. How many hops and kilometres does the cut add?

Functions to use: `shortest_path`, `path_weight`

Tip: `draw(H, [path1, path2])` shows two paths on the map of the network H.

In [ ]:
def exercise_1(G, D):
    """Paths from Sevilla to Valencia, before and after the cut of the fibre Madrid-Valencia."""
    # ---- TODO 1 ----
    # For G and for fiber_cut(G, "Madrid", "Valencia"), print the path with the fewest hops
    # (nx.shortest_path) and the shortest path in kilometres (weight="length") from Sevilla
    # to Valencia, with their number of hops and their length (km(H, path)).
    raise NotCompletedError("TODO 1: paths before and after the cut")

In [ ]:
run(exercise_1)

## Exercise 2

For each of the 8 single fibre cuts: does the network remain connected, and how many wavelengths does the fewest-hop routing of the lab traffic need? Which cut is the most harmful?

Functions to use: `is_strongly_connected`, `shortest_path_length`

In [ ]:
def exercise_2(G, D):
    """The effect of every single fibre cut on the fewest-hop bound of the lab traffic.

    For each cut: is the network still connected, and how many wavelengths does the
    fewest-hop routing of the lab traffic need?"""
    # ---- TODO 2 ----
    # For every fibre (u, v) in fibers(G): H = fiber_cut(G, u, v); check
    # nx.is_strongly_connected(H); if it is, print fewest_hop_bound(H, D) and its increase
    # with respect to fewest_hop_bound(G, D) = 408. Which cut increases the bound the most?
    raise NotCompletedError("TODO 2: fewest-hop bound after every single fibre cut")

In [ ]:
run(exercise_2)

## Exercise 3

The cut bound of section 5 for the other traffic matrix, `data/traffic/spain7_traffic.csv`: which set S is the most restrictive, and which capacity does it require?

Functions to use: `itertools.combinations`

In [ ]:
def exercise_3(G, D):
    """Cut bound (handout 2, section 5) for the other traffic matrix of Spain-7."""
    # ---- TODO 3 ----
    # Load data/traffic/spain7_traffic.csv, then repeat the computation of section 5 of
    # handout 2 (cell 5 of the tour): for every set S of nodes, the traffic from S to the
    # other nodes divided by the number of links leaving S, rounded up. Print the largest
    # value, the set S and the number of links.
    raise NotCompletedError("TODO 3: cut bound for spain7_traffic.csv")

In [ ]:
run(exercise_3)

## Exercise 4

With the fewest-hop routing of the lab traffic, how many wavelengths cross each node in transit (neither origin nor destination)?

Functions to use: `shortest_path`

In [ ]:
def exercise_4(G, D):
    """Transit traffic of every node with the fewest-hop routing of the lab traffic.

    Transit = wavelengths of the demands whose path crosses the node, without starting
    or ending there."""
    # ---- TODO 4 ----
    # For every demand d, path = nx.shortest_path(G, d.origin, d.destination); add
    # d.traffic to every intermediate node path[1:-1]. Print the nodes from the most
    # to the least loaded.
    raise NotCompletedError("TODO 4: transit traffic of every node")

In [ ]:
run(exercise_4)

## Exercise 5

Add Bilbao, linked to Madrid and Zaragoza with 40 wavelengths, generate 30 demands of 4 to 16 wavelengths with seed 3, and save both files. Open them in the graphical interface and run the splittable flow formulation. Is the problem feasible? Find the smallest feasible capacity with a sweep and compare it with the cut bound of exercise 3.

Functions to use: `add_city`, `add_link`, `random_traffic`

The files are written in the folder `results/lab1_exercises` (in Colab: *Files* panel on the left, right click > *Download*). Open them in PyNet2Plan on your computer with *File > Open topology...* and *Traffic > Open traffic matrix...*.

In [ ]:
def exercise_5(G, D):
    """A new city: Bilbao, linked to Madrid and Zaragoza; topology and traffic saved for the GUI."""
    # ---- TODO 5 ----
    # H = G.copy(); add the node with network.add_city(H, "Bilbao", allow_online=False) and
    # two bidirectional links of 40 wavelengths with network.add_link(H, "Bilbao", "Madrid",
    # capacity=40, bidirectional=True) (and the same towards Zaragoza). Generate
    # traffic.random_traffic(H, 30, 4, 16, seed=3), save the topology and the traffic in OUT
    # with network.save_topology and traffic.save_traffic, then open both files in the
    # graphical interface and run the splittable flow formulation.
    raise NotCompletedError("TODO 5: Spain-8 with Bilbao")

In [ ]:
run(exercise_5)

## Exercise 6 (optional)

Generate the first connected Waxman network with 15 nodes (section 6) and open it in the graphical interface.

Functions to use: `waxman_graph`, `from_networkx`

`waxman15.json` is written in the same folder, to be opened in the interface.

In [ ]:
def exercise_6(G, D):
    """(Optional) A connected Waxman network with 15 nodes, saved for the GUI."""
    # ---- TODO 6 ----
    # As in cell 6 of the tour: nx.waxman_graph(15, beta=0.6, alpha=0.3, seed=seed) for
    # seed = 0, 1, 2, ... until nx.is_connected; convert it with network.from_networkx and
    # save it in OUT.
    raise NotCompletedError("TODO 6: a connected Waxman network")

In [ ]:
run(exercise_6)

## Exercise 7 (optional)

Number of nodes and links, links per node and diameter in hops of Spain-7, NSFNET-14, COST239 and GEANT2.

Functions to use: `diameter`

In [ ]:
def exercise_7(G, D):
    """(Optional) Size of some real networks: nodes, links, average out-degree, diameter in hops."""
    # ---- TODO 7 ----
    # For spain7.json, nsfnet14.json, cost239.json and real/top22_geant2.json in
    # data/topologies: number of nodes and links, links per node, nx.diameter(H).
    raise NotCompletedError("TODO 7: size of some real networks")

In [ ]:
run(exercise_7)

## When you have finished

Compare the results of exercise 2 with the fibre cuts of handout 3 (section 5), where the capacity
of the links is limited to 60 wavelengths. The same functions are in `labs/lab1_exercises.py`,
which can be run from a terminal with `python labs/lab1_exercises.py`.